# YT Policy Change Detection

To obtain the most recent set of optimization report:
- Make a copy of the previous week's folder
    - Ensure that this Jupyter Notebook is in the folder
- In the copy, delete files ending with "Old" and change files ending with "New" to "Old"
- Run SQL queries and name them accordingly, ending with "CI/ISRC New"
    - ***SR Asset Top 1000:** "YT Sound Recording Asset T1000 CI New", "YT Sound Recording Asset T1000 ISRC New"*
    - ***SR Asset 2024:** "YT Sound Recording Asset 2024 CI New", "YT Sound Recording Asset 2024 ISRC New"*
    - ***SR Asset 2025:** "YT Sound Recording Asset 2025 CI New", "YT Sound Recording Asset 2025 ISRC New"*
    - ***Video Asset Top 1000:** "YT Video Assets T1000 New"*
    - ***Video Asset 2024:** "YT Video Assets 2024 CCID New", "YT Video Assets 2024 ISRC New"*
    - ***Video Asset 2025:** "YT Video Assets 2025 CCID New", "YT Video Assets 2025 ISRC New"*
    - ***Video MV Asset Change:** "YT MV to Web Asset Change Detection Old", "YT MV to Web Asset Change Detection New"*
- Change name of reports to be downloaded
    - *Ex) "YT_Asset_T1000_Report_**W8**.xlsx" --> "YT_Asset_T1000_Report_**W9**.xlsx"*
- Run all on this file

### Functions

In [10]:
# Codes adapted from original Change Detection Jupyter Notebooks

####################################################################################################################################

import pandas as pd
import os
import numpy as np
import shutil
import re

# Helper Functions
def sort_columns_by_prefix(df):

    def get_prefix(col):
        return next((col.split(s)[0] for s in ['_change', '_week1', '_week2'] if s in col), col)

    prefixes = [] 
    for col in df.columns:
        prefix = get_prefix(col)
        if prefix not in prefixes:
            prefixes.append(prefix)

    sorted_cols = []
    for prefix in prefixes:
        cols = [c for c in df.columns if get_prefix(c) == prefix]
        sorted_cols.extend(sorted(c for c in cols if '_change' in c) +
                           sorted(c for c in cols if '_week1' in c) +
                           sorted(c for c in cols if '_week2' in c) +
                           sorted(c for c in cols if '_change' not in c and '_week1' not in c and '_week2' not in c)) #added other cols
    return df[sorted_cols]


def highlight_changes(val):
    if isinstance(val, bool):
        color = 'red' if val else 'white'
    elif isinstance(val, (int, float)):  # Check for numeric types
        color = 'red' if val < 0 else 'green' if val > 0 else 'white'
    else:
        color = 'white' #Default color for other types

    return f'background-color: {color}'


def split_policy_info(df, policy_column='POLICY'):

    def extract_countries(text, keyword):
        if not isinstance(text, str):
            return ""
        match = re.search(f"{keyword}: (.*?)\\.", text)
        return match.group(1) if match else ""

    df['MONETIZE'] = df[policy_column].apply(lambda x: extract_countries(x, "Monetize the following countries"))
    df = df.drop(columns=['POLICY'])    
     
    return df


def compute_policy_changes(row):

    try: # Handle potential errors in policy strings
        countries_week1 = set(re.split(r', |\. ', row['MONETIZE_week1'] or "")) #Handle None values
        countries_week2 = set(re.split(r', |\. ', row['MONETIZE_week2'] or ""))

        dropped = countries_week1 - countries_week2
        added = countries_week2 - countries_week1

        return pd.Series([', '.join(dropped) or 'None', ', '.join(added) or 'None']) #More concise way to handle None

    except (TypeError, AttributeError): #Catch potential errors
        return pd.Series(['Error', 'Error'])  # Or handle differently



####################################################################################################################################

# Generating Report for Top 1000 Sound Recording Assets
def yt_asset_t1000(ccid_1, isrc_1, ccid_2, isrc_2):
    # Handling duplicates and merge
    ccid_1 = ccid_1.drop_duplicates(subset=["ASSET_ID"])
    isrc_1 = isrc_1.drop_duplicates(subset=["ASSET_ID"])

    mv_1 = pd.concat([ccid_1, isrc_1])

    ccid_2 = ccid_2.drop_duplicates(subset=["ASSET_ID"])
    isrc_2 = isrc_2.drop_duplicates(subset=["ASSET_ID"])

    mv_2 = pd.concat([ccid_2, isrc_2])

    # Merging the two datasets
    merged = pd.merge(mv_1, mv_2, how="outer", on="ASSET_ID", suffixes=("_week1", "_week2"))
    
    for column in mv_1.columns:
        if column != 'ASSET_ID':
            # string comparison
            condition = (merged[f'{column}_week2'].notna() & merged[f'{column}_week1'].notna())
            merged[f'{column}_change'] = (merged[f'{column}_week2'] != merged[f'{column}_week1']) & condition

    # Sort columns
    merged_s = sort_columns_by_prefix(merged)

    merged_1 = merged_s.drop_duplicates(subset=["ASSET_ID"])

    # Highlight changes
    report = merged_1.style.applymap(highlight_changes, subset=[col for col in merged_1.columns if '_change' in col])

    return report

# Generating Report for 2024/2025 Sound Recording Assets
def yt_asset_year(ccid_1, isrc_1, ccid_2, isrc_2):
    # Handling duplicates and merge
    ccid_1 = ccid_1.drop_duplicates(subset=["ASSET_ID"])
    isrc_1 = isrc_1.drop_duplicates(subset=["ASSET_ID"])

    mv_1 = pd.concat([ccid_1, isrc_1])

    ccid_2 = ccid_2.drop_duplicates(subset=["ASSET_ID"])
    isrc_2 = isrc_2.drop_duplicates(subset=["ASSET_ID"])

    mv_2 = pd.concat([ccid_2, isrc_2])

    # Merging the two datasets
    merged = pd.merge(mv_1, mv_2, how="outer", on="ASSET_ID", suffixes=("_week1", "_week2"))

    for column in mv_1.columns:
        if column != 'ASSET_ID':
            # string comparison
            condition = (merged[f'{column}_week2'].notna() & merged[f'{column}_week1'].notna())
            merged[f'{column}_change'] = (merged[f'{column}_week2'] != merged[f'{column}_week1']) & condition

    # Sort columns
    merged_s = sort_columns_by_prefix(merged)
    merged_1 = merged_s.drop_duplicates(subset=["ASSET_ID"])

    return merged_1

# Generating Report for Top 1000 Video Assets
def yt_video_t1000(last_week_data, this_week_data):
    last_week = split_policy_info(last_week_data)
    this_week = split_policy_info(this_week_data)

    # Merge the two datasets
    merged1000 = pd.merge(last_week, this_week, how="outer", on="VIDEO_ID", suffixes=("_week1", "_week2"))
    merged1000.isnull().sum()

    def fill_for_video(group):
        return group.ffill().bfill()

    for col in merged1000.columns:
        if merged1000[col].isnull().any():
            merged1000[col] = merged1000.groupby('VIDEO_ID')[col].transform(fill_for_video)

    merged1000 = merged1000.drop_duplicates(subset=["VIDEO_ID"])

    for column in last_week.columns:
        if column != 'VIDEO_ID':
            # string comparison
            condition = (merged1000[f'{column}_week2'].notna() & merged1000[f'{column}_week1'].notna())
            merged1000[f'{column}_change'] = (merged1000[f'{column}_week2'] != merged1000[f'{column}_week1']) & condition

    merged1000 = sort_columns_by_prefix(merged1000)
    merged1000[['countries_dropped', 'countries_added']] = merged1000.apply(compute_policy_changes, axis=1)
    report = merged1000.style.applymap(highlight_changes, subset=[col for col in merged1000.columns if '_change' in col])

    return report

# Generating Report for 2024/2025 Video Assets
def yt_video_year(last_ccid_1, last_isrc_1, this_ccid_2, this_isrc_2):
    # Previous week
    ccid_1 = split_policy_info(last_ccid_1)
    isrc_1 = split_policy_info(last_isrc_1)

    ccid_1['LENGTH'] = pd.to_numeric(ccid_1['LENGTH'], errors='coerce')
    ccid_1 = ccid_1[ccid_1['LENGTH'] > 61]

    isrc_1['LENGTH'] = pd.to_numeric(isrc_1['LENGTH'], errors='coerce')
    isrc_1 = isrc_1[isrc_1['LENGTH'] > 61]

    # Handle duplicates and merge
    ccid_1 = ccid_1.drop_duplicates(subset=["VIDEO_ID"])
    isrc_1 = isrc_1.drop_duplicates(subset=["VIDEO_ID"])

    mv_1 = pd.concat([ccid_1, isrc_1])
    mv_1 = mv_1[mv_1['VIDEO_ID'].str.len() == 11]

    # Current week
    ccid_2 = split_policy_info(this_ccid_2)
    isrc_2 = split_policy_info(this_isrc_2)

    ccid_2['LENGTH'] = pd.to_numeric(ccid_2['LENGTH'], errors='coerce')
    ccid_2 = ccid_2[ccid_2['LENGTH'] > 61]

    isrc_2['LENGTH'] = pd.to_numeric(isrc_2['LENGTH'], errors='coerce')
    isrc_2 = isrc_2[isrc_2['LENGTH'] > 61]

    # Handle duplicates and merge
    ccid_2 = ccid_2.drop_duplicates(subset=["VIDEO_ID"])
    isrc_2 = isrc_2.drop_duplicates(subset=["VIDEO_ID"])

    mv_2 = pd.concat([ccid_2, isrc_2])
    mv_2 = mv_2[mv_2['VIDEO_ID'].str.len() == 11]

    # Merge the two datasets
    merged = pd.merge(mv_1, mv_2, how="outer", on="VIDEO_ID", suffixes=("_week1", "_week2"))

    for column in mv_1.columns:
        if column != 'VIDEO_ID':
            # string comparison
            condition = (merged[f'{column}_week2'].notna() & merged[f'{column}_week1'].notna())
            merged[f'{column}_change'] = (merged[f'{column}_week2'] != merged[f'{column}_week1']) & condition

    merged_dd = merged.drop_duplicates(subset=["VIDEO_ID"])
    merged_1 = sort_columns_by_prefix(merged_dd)

    merged_1[['countries_dropped', 'countries_added']] = merged_1.apply(compute_policy_changes, axis=1)
    
    return merged_1

# Generating Report for MV Video Assets
def yt_video_asset_change(last_week_data, this_week_data):

    # Merge the two datasets
    merged = pd.merge(last_week_data, this_week_data, how="outer", on="VIDEO_ID", suffixes=("_week1", "_week2"))

    for column in last_week_data.columns:
        if column != 'VIDEO_ID':
            # string comparison
            condition = (merged[f'{column}_week2'].notna() & merged[f'{column}_week1'].notna())
            merged[f'{column}_change'] = (merged[f'{column}_week2'] != merged[f'{column}_week1']) & condition

    merged_1 = merged.drop_duplicates(subset=["VIDEO_ID"])
    merged_s = sort_columns_by_prefix(merged_1)
    
    # Only report video assets where asset types changed from MV
    report = merged_s[(merged_s['ASSET_TYPE_change'] == True) & (merged_s['ASSET_TYPE_week1'] == 'MUSIC_VIDEO')]

    # Return more than one active reference IDs
    report['ACTIVE_REFERENCE_IDS_week2'] = np.where((report['ACTIVE_REFERENCE_IDS_week2'].str.split().str.len() == 1).any(), np.nan, report['ACTIVE_REFERENCE_IDS_week2'])

    return report

### Generating Reports

In [11]:
# YouTube Sound Recording Top 1000 Asset Report
old_sr_ci_t1000 = pd.read_csv("YT Sound Recording Asset T1000 CI Old.csv")
old_sr_isrc_t1000 = pd.read_csv("YT Sound Recording Asset T1000 ISRC Old.csv")
new_sr_ci_t1000 = pd.read_csv("YT Sound Recording Asset T1000 CI New.csv")
new_sr_isrc_t1000 = pd.read_csv("YT Sound Recording Asset T1000 ISRC New.csv")

yt_sr_t1000_rpt = yt_asset_t1000(old_sr_ci_t1000, old_sr_isrc_t1000, new_sr_ci_t1000, new_sr_isrc_t1000)
# Only change report name to save report to current directory
yt_sr_t1000_rpt.to_excel('YT_Asset_T1000_Report_W14.xlsx', index=False)
# If you want to save to a specific folder...
# yt_sr_t1000_rpt.to_excel(r"/Users/allison/Desktop/WMG Data/July 15 Weekly Report/Reports/YT_Asset_T1000_Report_W14.xlsx", index=False)

/var/folders/5h/qvbyfns97cd5h388x6mzqxrr0000gq/T/ipykernel_64822/2070537280.py:104: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  report = merged_1.style.applymap(highlight_changes, subset=[col for col in merged_1.columns if '_change' in col])


In [12]:
# YouTube Sound Recording 2024 Asset Report
old_sr_ci_2024 = pd.read_csv("YT Sound Recording Asset 2024 CI Old.csv")
old_sr_isrc_2024 = pd.read_csv("YT Sound Recording Asset 2024 ISRC Old.csv")
new_sr_ci_2024 = pd.read_csv("YT Sound Recording Asset 2024 CI New.csv")
new_sr_isrc_2024 = pd.read_csv("YT Sound Recording Asset 2024 ISRC New.csv")

yt_sr_2024_rpt = yt_asset_year(old_sr_ci_2024, old_sr_isrc_2024, new_sr_ci_2024, new_sr_isrc_2024)
# Only change report name to save report to current directory
# yt_sr_2024_rpt.to_csv('YT_Asset_2024_Report_W9.csv', index=False)
# If you want to save to a specific folder...
yt_sr_2024_rpt.to_csv(r"/Users/guillermo.negrete/Documents/Py Projects/Change Detection/Change Detection Documents/Exports/090925/YT_Asset_2024_Report_W15.csv", index=False)

In [13]:
# YouTube Sound Recording 2025 Asset Report
old_sr_ci_2025 = pd.read_csv("YT Sound Recording Asset 2025 CI Old.csv")
old_sr_isrc_2025 = pd.read_csv("YT Sound Recording Asset 2025 ISRC Old.csv")
new_sr_ci_2025 = pd.read_csv("YT Sound Recording Asset 2025 CI New.csv")
new_sr_isrc_2025 = pd.read_csv("YT Sound Recording Asset 2025 ISRC New.csv")

yt_sr_2025_rpt = yt_asset_year(old_sr_ci_2025, old_sr_isrc_2025, new_sr_ci_2025, new_sr_isrc_2025)
# Only change report name to save report to current directory
# yt_sr_2025_rpt.to_csv('YT_Asset_2025_Report_W6.csv', index=False)
# If you want to save to a specific folder...
yt_sr_2025_rpt.to_csv(r"/Users/guillermo.negrete/Documents/Py Projects/Change Detection/Change Detection Documents/Exports/090925/YT_Asset_2025_Report_W12.csv", index=False)

In [14]:
# YouTube Video Recording T1000 Asset Report
old_vid_t1000 = pd.read_csv("YT Video Assets T1000 Old.csv")
new_vid_t1000 = pd.read_csv("YT Video Assets T1000 New.csv")

yt_vid_t1000_rpt = yt_video_t1000(old_vid_t1000, new_vid_t1000)
# Only change report name to save report to current directory
# yt_vid_t1000_rpt.to_excel('YT_Video_T1000_Report_W5.xlsx', index=False)
# If you want to save to a specific folder...
yt_vid_t1000_rpt.to_excel(r"/Users/guillermo.negrete/Documents/Py Projects/Change Detection/Change Detection Documents/Exports/090925/YT_Video_T1000_Report_W11.xlsx", index=False)

/var/folders/5h/qvbyfns97cd5h388x6mzqxrr0000gq/T/ipykernel_64822/2070537280.py:146: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return group.ffill().bfill()
/var/folders/5h/qvbyfns97cd5h388x6mzqxrr0000gq/T/ipykernel_64822/2070537280.py:146: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return group.ffill().bfill()
/var/folders/5h/qvbyfns97cd5h388x6mzqxrr0000gq/T/ipykernel_64822/2070537280.py:146: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=Fa

In [15]:
# YouTube Video Recording 2024 Asset Report
old_vid_ci_2024 = pd.read_csv("YT Video Assets 2024 CCID Old.csv")
old_vid_isrc_2024 = pd.read_csv("YT Video Assets 2024 ISRC Old.csv")
new_vid_ci_2024 = pd.read_csv("YT Video Assets 2024 CCID New.csv")
new_vid_isrc_2024 = pd.read_csv("YT Video Assets 2024 ISRC New.csv")

yt_vid_2024_rpt = yt_video_year(old_vid_ci_2024, old_vid_isrc_2024, new_vid_ci_2024, new_vid_isrc_2024)
# Only change report name to save report to current directory
# yt_vid_2024_rpt.to_csv('YT_Video_2024_Report_W9.csv', index=False)
# If you want to save to a specific folder...
yt_vid_2024_rpt.to_csv(r"/Users/guillermo.negrete/Documents/Py Projects/Change Detection/Change Detection Documents/Exports/090925/YT_Video_2024_Report_W15.csv", index=False)

/var/folders/5h/qvbyfns97cd5h388x6mzqxrr0000gq/T/ipykernel_64822/2070537280.py:214: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged_1[['countries_dropped', 'countries_added']] = merged_1.apply(compute_policy_changes, axis=1)
/var/folders/5h/qvbyfns97cd5h388x6mzqxrr0000gq/T/ipykernel_64822/2070537280.py:214: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged_1[['countries_dropped', 'countries_added']] = merged_1.apply(compute_policy_changes, axis=1)


In [16]:
# YouTube Video Recording 2025 Asset Report
old_vid_ci_2025 = pd.read_csv("YT Video Assets 2025 CCID Old.csv")
old_vid_isrc_2025 = pd.read_csv("YT Video Assets 2025 ISRC Old.csv")
new_vid_ci_2025 = pd.read_csv("YT Video Assets 2025 CCID New.csv")
new_vid_isrc_2025 = pd.read_csv("YT Video Assets 2025 ISRC New.csv")

yt_vid_2025_rpt = yt_video_year(old_vid_ci_2025, old_vid_isrc_2025, new_vid_ci_2025, new_vid_isrc_2025)
# Only change report name to save report to current directory
# yt_vid_2025_rpt.to_csv('YT_Video_2025_Report_W8.csv', index=False)
# If you want to save to a specific folder...
yt_vid_2025_rpt.to_csv(r"/Users/guillermo.negrete/Documents/Py Projects/Change Detection/Change Detection Documents/Exports/090925/YT_Video_2025_Report_W14.csv", index=False)

/var/folders/5h/qvbyfns97cd5h388x6mzqxrr0000gq/T/ipykernel_64822/2070537280.py:214: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged_1[['countries_dropped', 'countries_added']] = merged_1.apply(compute_policy_changes, axis=1)
/var/folders/5h/qvbyfns97cd5h388x6mzqxrr0000gq/T/ipykernel_64822/2070537280.py:214: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged_1[['countries_dropped', 'countries_added']] = merged_1.apply(compute_policy_changes, axis=1)


In [17]:
# YouTube Video MV Asset Change Report (Takes ~1 min)
old_vid_asset = pd.read_csv("YT MV to Web Asset Change Detection Old.csv")
new_vid_asset = pd.read_csv("YT MV to Web Asset Change Detection New.csv")

yt_vid_asset_change = yt_video_asset_change(old_vid_asset, new_vid_asset)
# Only change report name to save report to current directory
# yt_vid_asset_change.to_excel('YT_Video_Asset_Change_Report_W1.xlsx', index=False)
# If you want to save to a specific folder...
yt_vid_asset_change.to_excel(r"/Users/guillermo.negrete/Documents/Py Projects/Change Detection/Change Detection Documents/Exports/090925/YT_Video_Asset_Change_Report_W1.xlsx", index=False)

/var/folders/5h/qvbyfns97cd5h388x6mzqxrr0000gq/T/ipykernel_64822/2679219617.py:2: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  old_vid_asset = pd.read_csv("YT MV to Web Asset Change Detection Old.csv")
/var/folders/5h/qvbyfns97cd5h388x6mzqxrr0000gq/T/ipykernel_64822/2679219617.py:3: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  new_vid_asset = pd.read_csv("YT MV to Web Asset Change Detection New.csv")
/var/folders/5h/qvbyfns97cd5h388x6mzqxrr0000gq/T/ipykernel_64822/2070537280.py:237: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  report['ACTIVE_REFERENCE_IDS_week2'] = np.where((report['ACTIVE_REFERENCE_IDS_week2'].str.split().str.len() 